In [12]:
import kagglehub
import os
import pandas as pd

# Pobranie danych
path = kagglehub.dataset_download("xvivancos/tweets-during-cavaliers-vs-warriors")
print("Path to dataset files:", path)

# Wyświetlenie plików
files = os.listdir(path)
print("Files in dataset:", files)

# Wczytanie CSV z poprawnym kodowaniem
for file in files:
    if file.endswith(".csv"):
        file_path = os.path.join(path, file)
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='latin1')  # lub 'ISO-8859-1', 'cp1252'
        
        print(f"\nWczytano plik: {file}")
        print(df.info())
        print(df.head())


Path to dataset files: /Users/ewatrebacz/.cache/kagglehub/datasets/xvivancos/tweets-during-cavaliers-vs-warriors/versions/24
Files in dataset: ['TweetsNBA.csv', 'TweetsNBA.json', 'locations.csv']

Wczytano plik: TweetsNBA.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51425 entries, 0 to 51424
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 51425 non-null  int64  
 1   text                       51425 non-null  object 
 2   retweet_count              51425 non-null  int64  
 3   favorite_count             51425 non-null  int64  
 4   favorited                  51425 non-null  bool   
 5   truncated                  51425 non-null  bool   
 6   id_str                     51425 non-null  int64  
 7   in_reply_to_screen_name    1058 non-null   object 
 8   source                     51425 non-null  object 
 9   retweeted                  51425 non-null  b

In [ ]:
import pandas as pd
import networkx as nx
import random

# Wczytanie danych
tweets_df = pd.read_csv("/Users/ewatrebacz/.cache/kagglehub/datasets/xvivancos/tweets-during-cavaliers-vs-warriors/versions/24/TweetsNBA.csv", encoding="latin1")

# Upewnij się, że kolumna istnieje
print("Kolumny w zbiorze:", tweets_df.columns)

# Użyj 'user_id_str' jako unikalnego identyfikatora użytkownika
G = nx.DiGraph()

# Dodaj użytkowników jako wierzchołki
for user in tweets_df['user_id_str'].unique():
    G.add_node(user)

# Dodaj sztuczne krawędzie — symulujemy followersów
for _, row in tweets_df.iterrows():
    user = row['user_id_str']
    followers = tweets_df['user_id_str'].dropna().sample(n=random.randint(1, 5)).tolist()
    for follower in followers:
        if follower != user:
            G.add_edge(user, follower)

# Inicjalizacja statusów
status = {node: "Tree" for node in G.nodes()}
I = set()
f0 = 0.8  # próg


# Forest-fire algorithm
for u in G.nodes():
    if status[u] == "Tree":
        user_info = tweets_df[tweets_df['user_id_str'] == u]
        if not user_info.empty:
            activeness = user_info['statuses_count'].values[0]
            fu = min(1.0, activeness / 10000)
        else:
            fu = 0.0

        if fu >= f0:
            status[u] = "Fire"
            I.add(u)

    if status[u] == "Fire":
        for v in G.successors(u):
            status[v] = "Fire"
            I.add(v)

print("Liczba wierzchołków:", G.number_of_nodes())
print("Liczba poinformowanych użytkowników (Fire):", len(I))


Kolumny w zbiorze: Index(['Unnamed: 0', 'text', 'retweet_count', 'favorite_count', 'favorited',
       'truncated', 'id_str', 'in_reply_to_screen_name', 'source', 'retweeted',
       'created_at', 'in_reply_to_status_id_str', 'in_reply_to_user_id_str',
       'lang', 'listed_count', 'verified', 'location', 'user_id_str',
       'description', 'geo_enabled', 'user_created_at', 'statuses_count',
       'followers_count', 'favourites_count', 'protected', 'user_url', 'name',
       'time_zone', 'user_lang', 'utc_offset', 'friends_count', 'screen_name',
       'country_code', 'country', 'place_type', 'full_name', 'place_name',
       'place_id', 'place_lat', 'place_lon', 'lat', 'lon', 'expanded_url',
       'url'],
      dtype='object')
Liczba wierzchołków: 39450
Liczba poinformowanych użytkowników (Fire): 38145


In [20]:
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import os

print("Wczytanie danych")
tweets_df = pd.read_csv("/Users/ewatrebacz/.cache/kagglehub/datasets/xvivancos/tweets-during-cavaliers-vs-warriors/versions/24/TweetsNBA.csv", encoding="latin1")
tweets_df.dropna(subset=['user_id_str'], inplace=True)

# Subset danych
sample_users = tweets_df['user_id_str'].dropna().unique()[:300]
tweets_df = tweets_df[tweets_df['user_id_str'].isin(sample_users)]


print("Budowanie grafu")
G = nx.DiGraph()
for user in tweets_df['user_id_str'].unique():
    G.add_node(user)

for _, row in tweets_df.iterrows():
    user = row['user_id_str']
    followers = tweets_df['user_id_str'].dropna().sample(n=random.randint(1, 3)).tolist()
    for follower in followers:
        if follower != user:
            G.add_edge(user, follower)

print("Inicjalizacja statusów")
status = {node: "Tree" for node in G.nodes()}
f0 = 0.6
frames_dir = "fire_frames"
os.makedirs(frames_dir, exist_ok=True)
I = set()
step = 0

# Ustal pozycję wierzchołków dla spójnej wizualizacji
pos = nx.spring_layout(G, seed=42)

print("file=orest-fire z krokową animacją")
for u in G.nodes():
    updated = False
    if status[u] == "Tree":
        user_info = tweets_df[tweets_df['user_id_str'] == u]
        if not user_info.empty:
            activeness = user_info['statuses_count'].values[0]
            print(activeness/10000)
            fu = min(1.0, activeness / 10000)
        else:
            fu = 0.0

        if fu >= f0:
            status[u] = "Fire"
            I.add(u)
            updated = True

    if status[u] == "Fire":
        for v in G.successors(u):
            if status[v] != "Fire":
                status[v] = "Fire"
                I.add(v)
                updated = True

    # Jeśli coś się zmieniło — zapisz klatkę
    if updated:
        colors = ['red' if status[n] == "Fire" else 'green' for n in G.nodes()]
        plt.figure(figsize=(10, 8))
        nx.draw(G, pos, node_color=colors, node_size=20, edge_color='gray', with_labels=False)
        plt.title(f"Krok {step} – Rozprzestrzenianie informacji")
        plt.savefig(f"{frames_dir}/frame_{step:03d}.png")
        plt.close()
        step += 1

print("Tworzenie GIF-a")
images = []
frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith(".png")])
for filename in frame_files:
    img_path = os.path.join(frames_dir, filename)
    images.append(imageio.imread(img_path))

imageio.mimsave("fire_spread.gif", images, duration=0.7)
print("✅ GIF zapisany jako 'fire_spread.gif'")


Wczytanie danych
Budowanie grafu
Inicjalizacja statusów
file=orest-fire z krokową animacją
0.586
0.22
0.0578
6.7308
1.0357
0.9996
0.0888
0.1497
0.2471
0.5992
0.6222
8.4547
2.8083
0.0319
1.7645
1.2419
0.5225
0.1679
0.0077
1.8986
0.0025
0.1602
3.352
3.9598
1.0103
2.9673
0.8498
0.0316
2.1669
2.0049
1.1891
5.4398
1.6588
0.1389
0.0596
2.686
0.0021
0.1939
0.2033
0.0132
1.229
0.0013
1.9928
3.1332
0.0359
5.9603
2.2218
0.294
2.5783
0.0208
0.3553
6.2343
4.1688
61.591
2.9853
0.0013
3.5565
5.386
1.8448
0.183
0.6949
2.7925
0.0029
0.2315
1.153
6.6592
0.9595
0.2441
1.731
0.3422
0.2306
10.6916
1.3286
13.688
0.0107
6.3456
4.5339
0.4999
7.2099
0.3642
0.8233
4.0175
0.158
0.3089
1.2259
0.8001
1.2773
10.2082
0.5953
0.7653
0.0599
0.4791
1.0813
0.6174
0.3019
0.1137
0.2853
0.1557
0.0136
0.9793
5.2541
Tworzenie GIF-a
✅ GIF zapisany jako 'fire_spread.gif'


In [ ]:
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import os

# === KROK 1: Wczytanie danych ===
tweets_df = pd.read_csv("/Users/ewatrebacz/.cache/kagglehub/datasets/xvivancos/tweets-during-cavaliers-vs-warriors/versions/24/TweetsNBA.csv", encoding="latin1")
tweets_df.dropna(subset=['user_id_str'], inplace=True)
#print(tweets_df['user_id_str'])

# Ogranicz do 300 użytkowników
sample_users = tweets_df['user_id_str'].dropna().unique()[:300]
tweets_df = tweets_df[tweets_df['user_id_str'].isin(sample_users)]

# === KROK 2: Budowa grafu ===
G = nx.DiGraph()
for user in tweets_df['user_id_str'].unique():
    G.add_node(user)

# Dodaj losowych followersów jako krawędzie
for _, row in tweets_df.iterrows():
    user = row['user_id_str']
    followers = tweets_df['user_id_str'].dropna().sample(n=1).tolist()
    for follower in followers:
        if follower != user:
            G.add_edge(user, follower)

# === KROK 3: Inicjalizacja ===
status = {node: "Tree" for node in G.nodes()}
S = set()    # spreaders
NS = set()   # non-spreaders
f0 = 0.6     # próg dla tweetowania
r0 = 0.5     # próg dla retweetowania

# Przygotuj folder do ramek
frames_dir = "mff_frames"
os.makedirs(frames_dir, exist_ok=True)

step = 0
pos = nx.spring_layout(G, seed=42)  # stała pozycja węzłów do rysowania

# === KROK 4: Algorytm Modified Forest Fire ===
for u in G.nodes():
    updated = False

    # Etap 1: czy u tweetuje?
    if status[u] == "Tree":
        user_info = tweets_df[tweets_df['user_id_str'] == u]
        fu = min(1.0, user_info['statuses_count'].values[0] / 10000) if not user_info.empty else 0.0

        if fu >= f0:
            status[u] = "Fire"
            S.add(u)
            updated = True

    # Etap 2: czy u rozprzestrzenia na swoich followersów?
    if status[u] == "Fire":
        for v in G.successors(u):
            if status[v] == "Tree":
                follower_info = tweets_df[tweets_df['user_id_str'] == v]
                ru = min(1.0, follower_info['favourites_count'].values[0] / 10000) if not follower_info.empty else 0.0

                if ru >= r0:
                    status[v] = "Fire"
                    S.add(v)
                else:
                    status[v] = "Burnt"
                    NS.add(v)
                updated = True

    # Etap 3: Zapisz klatkę co kilka kroków
    if updated and step % 3 == 0:
        colors = []
        for n in G.nodes():
            if status[n] == "Fire":
                colors.append("red")
            elif status[n] == "Burnt":
                colors.append("gray")
            else:
                colors.append("green")
        plt.figure(figsize=(10, 8))
        nx.draw(G, pos, node_color=colors, node_size=20, edge_color='lightgray', with_labels=False)
        plt.title(f"MFF – Krok {step}")
        plt.savefig(f"{frames_dir}/frame_{step:03d}.png")
        plt.close()
    step += 1

# === KROK 5: Tworzenie GIF-a ===
images = []
frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith(".png")])
for filename in frame_files:
    images.append(imageio.imread(os.path.join(frames_dir, filename)))

imageio.mimsave("mff_fire_spread.gif", images, duration=0.7)
print("✅ GIF zapisany jako 'mff_fire_spread.gif'")


✅ GIF zapisany jako 'mff_fire_spread.gif'


In [25]:
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import os
from datetime import datetime

# Wczytanie danych
df = pd.read_csv("/Users/ewatrebacz/.cache/kagglehub/datasets/xvivancos/tweets-during-cavaliers-vs-warriors/versions/24/TweetsNBA.csv", encoding="latin1")
df.dropna(subset=['user_id_str'], inplace=True)
df['user_created_at'] = pd.to_datetime(df['user_created_at'], errors='coerce')

# Ogranicz do 300 użytkowników
sample_users = df['user_id_str'].unique()[:1000]
df = df[df['user_id_str'].isin(sample_users)]

# Buduj graf
G = nx.DiGraph()
for user in df['user_id_str'].unique():
    G.add_node(user)

for _, row in df.iterrows():
    user = row['user_id_str']
    followers = df['user_id_str'].sample(n=1).tolist()
    for follower in followers:
        if follower != user:
            G.add_edge(user, follower)

# Wagi
w1, w2 = 0.5, 0.5  # dla fᵤ
w_IM = w_SS = w_UA = w_TS = 0.25  # dla rᵤ

# Funkcje pomocnicze
def calculate_UA(row):
    try:
        days = (datetime.now() - row['user_created_at']).days
        return row['statuses_count'] / days if days > 0 else 0
    except:
        return 0

def assign_TS(text):
    if "#NBAFinals" in str(text):
        return 4  # Trending + Global
    return 2  # domyślnie Non-trending + Global

def is_mentioned():
    return random.choice([0, 1])  # losowo

def similarity_score(u_row, v_row):
    loc_sim = 1 if u_row['location'] == v_row['location'] else 0
    lang_sim = 1 if u_row['user_lang'] == v_row['user_lang'] else 0
    return (loc_sim + lang_sim) / 2  # uproszczony SS

# Inicjalizacja
status = {n: "Tree" for n in G.nodes()}
S, NS = set(), set()
frames_dir = "mff_frames_full"
os.makedirs(frames_dir, exist_ok=True)
pos = nx.spring_layout(G, seed=42)
step = 0
f0, r0 = 0.6, 0.5

# Algorytm MFF z dokładnym liczeniem prawdopodobieństw
for u in G.nodes():
    u_info = df[df['user_id_str'] == u].iloc[0]

    UA_u = calculate_UA(u_info)
    TS_u = assign_TS(u_info['text'])
    fu = w1 * UA_u + w2 * TS_u
    print(fu)

    if status[u] == "Tree" and fu >= f0:
        status[u] = "Fire"
        S.add(u)

    if status[u] == "Fire":
        for v in G.successors(u):
            if status[v] == "Tree":
                v_info = df[df['user_id_str'] == v].iloc[0]
                UA_v = calculate_UA(v_info)
                TS_v = assign_TS(v_info['text'])
                IM = is_mentioned()
                SS = similarity_score(u_info, v_info)
                ru = w_IM*IM + w_SS*SS + w_UA*UA_v + w_TS*TS_v
                print(ru)

                if ru >= r0:
                    status[v] = "Fire"
                    S.add(v)
                else:
                    status[v] = "Burnt"
                    NS.add(v)

    if step % 3 == 0:
        color_map = []
        for n in G.nodes():
            if status[n] == "Fire":
                color_map.append("red")
            elif status[n] == "Burnt":
                color_map.append("gray")
            else:
                color_map.append("green")
        plt.figure(figsize=(10, 8))
        nx.draw(G, pos, node_color=color_map, node_size=20, edge_color='lightgray', with_labels=False)
        plt.title(f"Step {step}")
        plt.savefig(f"{frames_dir}/frame_{step:03d}.png")
        plt.close()
    step += 1

# Tworzenie GIF-a
images = [imageio.imread(f"{frames_dir}/" + f) for f in sorted(os.listdir(frames_dir)) if f.endswith(".png")]
imageio.mimsave("mff_final.gif", images, duration=0.7)
print("✅ GIF saved as mff_final.gif")


/var/folders/cb/vb_vwtjs4lb6ny7_6h2tg1v00000gn/T/ipykernel_92628/2107115375.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at'] = pd.to_datetime(df['user_created_at'], errors='coerce')


✅ GIF saved as mff_final.gif
